# Exercise 3.5 — The SWAP Trick in Action

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Section 3.3: The SWAP Operator and Purity*

---

## Motivation

The SWAP trick $\mathrm{Tr}(AB) = \mathrm{Tr}[(A \otimes B)F]$ is the computational engine behind purity estimation, Rényi entropy measurements, and the Weingarten calculus.  In experiments, it allows measuring the purity of a quantum state (a nonlinear function of $\rho$) via **two-copy interference** — no tomography needed.  This exercise develops fluency with the SWAP trick and connects it to Haar-averaged entanglement.

## Exercise Statement

**(a)** Prove $\mathrm{Tr}[(A \otimes B) F] = \mathrm{Tr}(AB)$ for the SWAP operator $F$.

**(b)** For the Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$, compute $\rho_A$ and verify $\mathrm{Tr}(\rho_A^2) = \mathrm{Tr}[(\rho_A \otimes \rho_A) F_A] = 1/2$.

**(c)** Compare with the Haar-averaged purity $\mathbb{E}[\gamma] = (m+n)/(mn+1)$ at $m = n = 2$.  Is the Bell state more or less entangled than a typical random state?

## Solution

### Part (a): Proof of the SWAP trick

The SWAP operator acts as $F|i,j\rangle = |j,i\rangle$.  Therefore:

$$
\mathrm{Tr}[(A \otimes B)F] = \sum_{i,j} \langle i,j|(A \otimes B)|j,i\rangle = \sum_{i,j} A_{ij}\, B_{ji} = \mathrm{Tr}(AB). \quad \checkmark
$$

The SWAP converts a two-copy tensor product into a single-copy trace — trading one quantum copy for a classical computation.

### Part (b): Bell state purity

**Step 1:** The reduced state of the Bell state is:

$$
\rho_A = \mathrm{Tr}_B|\Phi^+\rangle\langle\Phi^+| = \frac{1}{2}(|0\rangle\langle 0| + |1\rangle\langle 1|) = \frac{I}{2}.
$$

**Step 2 (direct):** $\mathrm{Tr}(\rho_A^2) = \mathrm{Tr}(I/4) = 2/4 = 1/2.$

**Step 3 (SWAP trick):**

$$
\rho_A \otimes \rho_A = \frac{I}{2} \otimes \frac{I}{2} = \frac{I_4}{4}.
$$

$$
\mathrm{Tr}[(\rho_A \otimes \rho_A) F] = \frac{1}{4}\mathrm{Tr}(F) = \frac{1}{4} \cdot 2 = \frac{1}{2}. \quad \checkmark
$$

Both methods agree: $\gamma = 1/2 = 1/D_A$, confirming the Bell state is maximally entangled.

### Part (c): Comparison with Haar average

The Haar-averaged purity for a bipartite system with subsystem dimensions $m$ and $n$ is:

$$
\mathbb{E}[\gamma] = \frac{m + n}{mn + 1}.
$$

For $m = n = 2$ (two qubits on a 4-dimensional Hilbert space):

$$
\mathbb{E}[\gamma] = \frac{2 + 2}{2 \cdot 2 + 1} = \frac{4}{5} = 0.8.
$$

The Bell state has $\gamma_{\mathrm{Bell}} = 0.5 < 0.8 = \mathbb{E}[\gamma]$.  The Bell state is **more entangled** than a typical Haar-random state on 2 qubits.

This makes sense: at $D = 4$, the Page curve prediction is approximate, and the typical state is not yet close to maximal entanglement.  Only for $mn \gg 1$ does $\mathbb{E}[\gamma] \to 1/m$ (Page's result), which would match the maximally entangled value.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp
from sympy import Matrix, trace, Symbol, I, eye, TensorProduct, simplify

# The SWAP trick: Tr(A*B) = Tr(SWAP * (A tensor B))
# Verify for 2x2 matrices
a, b, c, d = sp.symbols('a b c d')
e, f, g, h = sp.symbols('e f g h')

A = Matrix([[a, b], [c, d]])
B = Matrix([[e, f], [g, h]])

# Direct computation
tr_AB = trace(A * B)
print(f'Tr(A*B) = {sp.expand(tr_AB)}')

# SWAP operator on C^2 tensor C^2
SWAP = Matrix([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]])

# A tensor B as 4x4
AtB = TensorProduct(A, B)
swap_result = trace(SWAP * AtB)
print(f'Tr(SWAP * (A x B)) = {sp.expand(swap_result)}')

# Verify equality
assert sp.expand(tr_AB - swap_result) == 0
print('\nSWAP trick verified: Tr(AB) = Tr(SWAP * (A tensor B))')

# Application: purity Tr(rho^2) = Tr(SWAP * rho^{tensor 2})
print('Application: Tr(rho^2) = Tr(SWAP * rho x rho)')
print('This connects purity to the 2-copy Hilbert space.')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

# Part (b): Verify via SWAP trick and direct computation
D = 2
F = np.zeros((D**2, D**2))
for a in range(D):
    for b in range(D):
        F[a*D+b, b*D+a] = 1

# Bell state purity
rho_A = np.eye(D) / D
purity_direct = np.trace(rho_A @ rho_A).real
purity_swap = np.trace(np.kron(rho_A, rho_A) @ F).real
print(f"Bell purity (direct): {purity_direct:.4f}")
print(f"Bell purity (SWAP):   {purity_swap:.4f}")
assert np.isclose(purity_direct, 0.5)
assert np.isclose(purity_swap, 0.5)
print()

In [ ]:
# Part (c): Haar-averaged purity
m, n = 2, 2
D_total = m * n
n_samples = 20000

purities = []
for _ in range(n_samples):
    psi = unitary_group.rvs(D_total)[:, 0]
    rho = np.outer(psi, psi.conj()).reshape(m, n, m, n)
    rho_A = np.trace(rho, axis1=1, axis2=3)
    purities.append(np.trace(rho_A @ rho_A).real)

E_gamma = np.mean(purities)
pred = (m + n) / (m*n + 1)
print(f"E[γ] = {E_gamma:.4f}  (theory (m+n)/(mn+1) = {pred:.4f})")
print(f"Bell purity = 0.500 < E[γ] = {pred:.3f}: Bell is MORE entangled than typical.")
assert abs(E_gamma - pred) < 0.02
print("\nSWAP trick and Haar average verified. ✓")